In [4]:
import os
from geopy import distance
import pandas as pd

def calculate_distance(row):
    city_coordinates = (row['lat'], row['lng'])
    return distance.geodesic(city_coordinates, home_city_coordinates).km

data_pkg_path = '../data'
filename = 'worldcities.csv'
path = os.path.join(data_pkg_path, filename)
df = pd.read_csv(path)

home_country = 'India'
home_city = 'Bengaluru'

filtered = df[df['country'] == home_country]
country_df = df[df['country'] == home_country].copy()
filtered = country_df[country_df['city_ascii'] == home_city]
home_city_coordinates = (filtered.iloc[0]['lat'], filtered.iloc[0]['lng'])


result = country_df.apply(calculate_distance, axis=1)
country_df['distance'] = result
filtered = country_df[['city_ascii','distance']]

# Let's remove the row with home_city
filtered = filtered[filtered['city_ascii'] != home_city]

filtered = filtered.rename(columns = {'city_ascii': 'city'})
output_filename = 'pandas_exercise.csv'
output_dir = '../output'
output_path = os.path.join(output_dir, output_filename)
filtered.to_csv(output_path, index=False)
print('Successfully written output file at {}'.format(output_path))

Successfully written output file at ../output/pandas_exercise.csv


Prompt for Claude AI

In the country_df, I want to add a new column 'elevation' using the Open Meteo Elevation API https://open-meteo.com/en/docs/elevation-api. Use this API to add an `elevation` column for each city in your country using the `country_df` dataframe.

In [ ]:
import requests
import numpy as np

import os
from geopy import distance
import pandas as pd

def calculate_distance(row):
    city_coordinates = (row['lat'], row['lng'])
    return distance.geodesic(city_coordinates, home_city_coordinates).km

data_pkg_path = '../data'
filename = 'worldcities.csv'
path = os.path.join(data_pkg_path, filename)
df = pd.read_csv(path)

home_country = 'India'

country_df = df[df['country'] == home_country].copy()

def get_elevations(lats, lngs, batch_size=100):
    elevations = []
    for i in range(0, len(lats), batch_size):
        batch_lats = lats[i:i+batch_size]
        batch_lngs = lngs[i:i+batch_size]
        params = {
            'latitude': ','.join(map(str, batch_lats)),
            'longitude': ','.join(map(str, batch_lngs)),
        }
        response = requests.get('https://api.open-meteo.com/v1/elevation', params=params)
        response.raise_for_status()
        elevations.extend(response.json()['elevation'])
    return elevations

lats = country_df['lat'].tolist()
lngs = country_df['lng'].tolist()

country_df['elevation'] = get_elevations(lats, lngs)
country_df